# 🔬 Data Mining — Classification: Advanced Techniques
## Lecture 2 Applied Notebook
**Haydar Kılıç | Artificial Intelligence Engineering**

---
This notebook covers the following topics with hands-on implementation:

1. **K-Nearest Neighbors (KNN)**
2. **Naive Bayes Classifier (NBC)**
3. **Artificial Neural Networks (ANN) — Perceptron & Multi-Layer**
4. **Support Vector Machines (SVM)**

In [ ]:
# ========================================================
# IMPORT REQUIRED LIBRARIES
# ========================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

# Sklearn modules
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.datasets import make_classification, make_circles, make_moons

print('✅ All libraries loaded successfully!')
print(f'NumPy: {np.__version__} | Pandas: {pd.__version__}')

---
# SECTION 1: K-Nearest Neighbors Classifier (KNN)

## 📖 Theoretical Summary
KNN is a **lazy learner** algorithm. Instead of building a model during training, it stores the entire dataset in memory. When predicting:

$$y' = \\arg\\max_v \\sum_{(x_i, y_i) \\in D_z} I(v = y_i)$$

**Euclidean Distance:**
$$d(x, y) = \\sqrt{\\sum_{i=1}^{n}(x_i - y_i)^2}$$

**Distance-Weighted Voting:**
$$y' = \\arg\\max_v \\sum_{(x_i, y_i) \\in D_z} w_i \\cdot I(v = y_i), \\quad w_i = \\frac{1}{d(x', x_i)^2}$$

## 1.1 Step-by-Step Implementation of the Lecture Slide Example
**Problem:** Predicting student performance based on Math and Physics grades

In [ ]:
# ========================================================
# FROM-SCRATCH IMPLEMENTATION OF THE SLIDE EXAMPLE
# ========================================================

# Training data
data = {
    'Student':  ['A', 'B', 'C', 'D', 'E'],
    'Math':     [70, 60, 40, 30, 55],
    'Physics':  [80, 65, 50, 35, 60],
    'Class':    [1,   1, -1, -1,  1]   # 1: Pass, -1: Fail
}
df = pd.DataFrame(data)
print('=== TRAINING DATA ===')
df_display = df.copy()
df_display['Class'] = df_display['Class'].map({1: '✓ Pass', -1: '✗ Fail'})
print(df_display.to_string(index=False))

# Test point
x_new = np.array([50, 55])
print(f'\n🎯 New Student: Math={x_new[0]}, Physics={x_new[1]}')

# Compute Euclidean distances
X_train = df[['Math', 'Physics']].values
distances = []
for i, row in df.iterrows():
    x = np.array([row['Math'], row['Physics']])
    d = np.sqrt(np.sum((x_new - x)**2))
    distances.append(d)

df['Distance'] = distances
df_sorted = df.sort_values('Distance').reset_index(drop=True)

print('\n=== DISTANCES SORTED ASCENDING ===')
df_sorted_display = df_sorted.copy()
df_sorted_display['Class'] = df_sorted_display['Class'].map({1: '✓', -1: '✗'})
df_sorted_display['Distance'] = df_sorted_display['Distance'].round(2)
print(df_sorted_display[['Student', 'Math', 'Physics', 'Distance', 'Class']].to_string(index=False))

In [ ]:
# ========================================================
# CLASSIFICATION DECISION FOR k=3
# ========================================================
k = 3
k_neighbors = df_sorted.head(k)

print(f'=== k={k} NEAREST NEIGHBORS ===')
for _, row in k_neighbors.iterrows():
    class_str = '✓ Pass' if row['Class'] == 1 else '✗ Fail'
    print(f"  {row['Student']}: d={row['Distance']:.2f} → {class_str}")

# Majority voting
votes = k_neighbors['Class'].values
n_pass = np.sum(votes == 1)
n_fail = np.sum(votes == -1)
print(f'\n📊 Vote Count: ✓ Pass = {n_pass} | ✗ Fail = {n_fail}')
prediction = '✓ PASS' if n_pass > n_fail else '✗ FAIL'
print(f'🏆 Decision: New student → {prediction}')

print('\n=== DISTANCE-WEIGHTED VOTING (Optional) ===')
total_weight = {1: 0.0, -1: 0.0}
for _, row in k_neighbors.iterrows():
    w = 1.0 / row['Distance']
    total_weight[row['Class']] += w
    class_str = '✓' if row['Class'] == 1 else '✗'
    print(f"  {row['Student']}: w = 1/{row['Distance']:.2f} = {w:.4f} → {class_str}")

print(f'\n  S(✓) = {total_weight[1]:.4f}')
print(f'  S(✗) = {total_weight[-1]:.4f}')
weighted_pred = '✓ PASS' if total_weight[1] > total_weight[-1] else '✗ FAIL'
print(f'🏆 Weighted Decision: New student → {weighted_pred}')

In [ ]:
# ========================================================
# VISUALIZATION: EFFECT OF k
# ========================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('KNN — Effect of k on Decision Boundary', fontsize=14, fontweight='bold')

colors = {1: '#2ecc71', -1: '#e74c3c'}
labels = {1: '✓ Pass', -1: '✗ Fail'}

for ax_idx, k_val in enumerate([1, 3, 5]):
    ax = axes[ax_idx]
    
    # Decision grid
    xx, yy = np.meshgrid(np.linspace(20, 85, 200), np.linspace(25, 90, 200))
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    
    # Manual KNN
    def knn_predict_manual(test_point, X, y, k):
        dists = np.sqrt(np.sum((X - test_point)**2, axis=1))
        idx = np.argsort(dists)[:k]
        votes = y[idx]
        return 1 if np.sum(votes == 1) >= np.sum(votes == -1) else -1
    
    X_tr = df[['Math', 'Physics']].values
    y_tr = df['Class'].values
    
    Z = np.array([knn_predict_manual(p, X_tr, y_tr, k_val) for p in grid_points])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#fadbd8', '#d5f5e3']))
    ax.contour(xx, yy, Z, colors='gray', linewidths=0.8, linestyles='--')
    
    # Training points
    for _, row in df.iterrows():
        ax.scatter(row['Math'], row['Physics'],
                   c=colors[row['Class']], s=120, edgecolors='black', zorder=5)
        ax.annotate(row['Student'], (row['Math']+0.5, row['Physics']+0.5), fontsize=9)
    
    # Test point
    ax.scatter(*x_new, c='blue', marker='*', s=250, zorder=6, label='New Student')
    
    # k-neighbor circle
    dists_sorted = sorted([(np.sqrt(np.sum((x_new - X_tr[i])**2)), i) for i in range(len(X_tr))])
    r = dists_sorted[k_val-1][0] + 0.5
    circle = plt.Circle(x_new, r, color='blue', fill=False, linestyle=':', linewidth=1.5)
    ax.add_patch(circle)
    
    ax.set_title(f'k = {k_val}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Math Grade')
    ax.set_ylabel('Physics Grade')
    green_patch = mpatches.Patch(color='#2ecc71', label='Pass')
    red_patch = mpatches.Patch(color='#e74c3c', label='Fail')
    ax.legend(handles=[green_patch, red_patch], loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()
print('💡 Small k → flexible decision boundary (overfitting risk)')
print('💡 Large k → smooth decision boundary (underfitting risk)')

## 1.2 KNN with Sklearn — On Real Data

In [ ]:
# ========================================================
# SKLEARN KNN — IRIS DATASET
# ========================================================
from sklearn.datasets import load_iris

iris = load_iris()
X, y = iris.data[:, :2], iris.target  # First 2 features (for visualization)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Accuracy for different k values
k_values = range(1, 21)
accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test))
    accuracies.append(acc)

# Best k
best_k = k_values[np.argmax(accuracies)]
print(f'✅ Best k = {best_k} | Accuracy = {max(accuracies):.2%}')

# Plot
plt.figure(figsize=(10, 4))
plt.plot(k_values, accuracies, 'bo-', linewidth=2, markersize=8)
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best k={best_k}')
plt.xlabel('k Value', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('KNN: Effect of k on Accuracy (Iris Dataset)', fontsize=13)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Best model report
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X_train, y_train)
y_pred = knn_best.predict(X_test)
print('\n=== CLASSIFICATION REPORT ===')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

---
# SECTION 2: Naive Bayes Classifier (NBC)

## 📖 Theoretical Summary
Naive Bayes is based on the **conditional independence** assumption of features:

$$P(x|y) = \\prod_{i=1}^{d} P(x_i|y)$$

Posterior probability (Bayes' Theorem):

$$P(y|x) \\propto P(y) \\prod_{i=1}^{d} P(x_i|y)$$

**Decision:** Choose the class with the highest posterior probability.

In [ ]:
# ========================================================
# NBC SLIDE EXAMPLE IMPLEMENTATION
# Predicting Loan Default
# ========================================================

# Training data
nbc_data = pd.DataFrame({
    'Tid': range(1, 11),
    'HomeOwner':    ['Yes','No','No','Yes','No','No','Yes','No','No','No'],
    'MaritalStatus':['Single','Married','Single','Married','Divorced','Married','Divorced','Single','Married','Single'],
    'AnnualIncome': [125, 100, 70, 120, 95, 60, 220, 85, 75, 90],
    'Defaulted':    ['No','No','No','No','Yes','No','No','Yes','No','Yes']
})

print('=== TRAINING DATA ===')
print(nbc_data.to_string(index=False))

# Prior probabilities
P_yes = (nbc_data['Defaulted'] == 'Yes').mean()
P_no  = (nbc_data['Defaulted'] == 'No').mean()
print(f'\n📊 Prior Probabilities:')
print(f'  P(Yes) = {P_yes:.1f}')
print(f'  P(No)  = {P_no:.1f}')

In [ ]:
# ========================================================
# CONDITIONAL PROBABILITIES BY CLASS
# ========================================================
yes_df = nbc_data[nbc_data['Defaulted'] == 'Yes']
no_df  = nbc_data[nbc_data['Defaulted'] == 'No']

print('=== "YES" CLASS (Defaulted Borrowers) ===')
print(yes_df[['HomeOwner','MaritalStatus','AnnualIncome']].to_string(index=False))

# P(xi|y) for categorical features
def conditional_prob(df_class, column, value):
    return (df_class[column] == value).sum() / len(df_class)

# Test instance: HomeOwner=No, Married, Income=120K
print('\n=== TEST INSTANCE: HomeOwner=No, Married, Income=120K ===')

# For No class
P_ho_no_no = conditional_prob(no_df, 'HomeOwner', 'No')
P_ms_m_no  = conditional_prob(no_df, 'MaritalStatus', 'Married')

# Gaussian for continuous attribute
mu_no  = no_df['AnnualIncome'].mean()
std_no = no_df['AnnualIncome'].std(ddof=1)
from scipy.stats import norm
P_inc_no = norm.pdf(120, mu_no, std_no)

P_x_no = P_ho_no_no * P_ms_m_no * P_inc_no
print(f'\n"No" Class:')
print(f'  P(HomeOwner=No | No)       = {P_ho_no_no:.4f}  [{int(P_ho_no_no*len(no_df))}/{len(no_df)}]')
print(f'  P(Married | No)             = {P_ms_m_no:.4f}  [{int(P_ms_m_no*len(no_df))}/{len(no_df)}]')
print(f'  P(Income=120K | No) [Gauss] = {P_inc_no:.6f}  (μ={mu_no:.1f}, σ={std_no:.1f})')
print(f'  ──────────────────────────────────')
print(f'  P(x|No) = {P_x_no:.8f}')

# For Yes class
P_ho_no_yes = conditional_prob(yes_df, 'HomeOwner', 'No')
P_ms_m_yes  = conditional_prob(yes_df, 'MaritalStatus', 'Married')
mu_yes  = yes_df['AnnualIncome'].mean()
std_yes = yes_df['AnnualIncome'].std(ddof=1)
P_inc_yes = norm.pdf(120, mu_yes, std_yes)
P_x_yes = P_ho_no_yes * P_ms_m_yes * P_inc_yes

print(f'\n"Yes" Class:')
print(f'  P(HomeOwner=No | Yes)       = {P_ho_no_yes:.4f}')
print(f'  P(Married | Yes)            = {P_ms_m_yes:.4f}  ← Zero Probability!')
print(f'  P(Income=120K | Yes) [Gauss]= {P_inc_yes:.2e}')
print(f'  P(x|Yes) = {P_x_yes:.8f}')

print('\n=== POSTERIOR PROBABILITIES (Unnormalized) ===')
score_no  = P_no  * P_x_no
score_yes = P_yes * P_x_yes
print(f'  P(No)  × P(x|No)  = {score_no:.8f}')
print(f'  P(Yes) × P(x|Yes) = {score_yes:.8f}')
decision = 'No (Will Not Default)' if score_no > score_yes else 'Yes (Will Default)'
print(f'\n🏆 DECISION: {decision}')

In [ ]:
# ========================================================
# SKLEARN GaussianNB — VERIFICATION
# ========================================================
from sklearn.preprocessing import LabelEncoder

# Encoding
le_ho = LabelEncoder(); le_ms = LabelEncoder(); le_d = LabelEncoder()
X_nbc = np.column_stack([
    le_ho.fit_transform(nbc_data['HomeOwner']),
    le_ms.fit_transform(nbc_data['MaritalStatus']),
    nbc_data['AnnualIncome'].values
])
y_nbc = le_d.fit_transform(nbc_data['Defaulted'])

gnb = GaussianNB()
gnb.fit(X_nbc, y_nbc)

x_test_enc = np.array([[
    le_ho.transform(['No'])[0],
    le_ms.transform(['Married'])[0],
    120
]])

pred  = gnb.predict(x_test_enc)[0]
proba = gnb.predict_proba(x_test_enc)[0]

print('=== Verification with sklearn GaussianNB ===')
print(f'Predicted Class : {le_d.inverse_transform([pred])[0]}')
print(f'P(No)  = {proba[le_d.transform(["No"])[0]]:.6f}')
print(f'P(Yes) = {proba[le_d.transform(["Yes"])[0]]:.6f}')
print('\n✅ Same result as our manual calculation!')

---
# SECTION 3: Artificial Neural Networks (ANN)

## 📖 Theoretical Summary

### Perceptron
$$\\hat{y} = \\text{sign}(\\tilde{w}^T \\tilde{x}) = \\begin{cases} 1 & \\text{if } w^Tx + b > 0 \\\\ -1 & \\text{otherwise} \\end{cases}$$

### Weight Update Rule
$$w_j^{(k+1)} = w_j^{(k)} + \\lambda(y_i - \\hat{y}_i^{(k)}) x_{ij}$$

### Multi-Layer Network — Activation
$$a_i^l = f(z_i^l) = f\\left(\\sum_j w_{ij}^l a_j^{l-1} + b_i^l\\right)$$

### Sigmoid Function
$$\\sigma(z) = \\frac{1}{1 + e^{-z}}$$

In [ ]:
# ========================================================
# ACTIVATION FUNCTIONS VISUALIZATION
# ========================================================
z = np.linspace(-4, 4, 300)

activations = {
    'Linear':       z,
    'Sigmoid (σ)':  1 / (1 + np.exp(-z)),
    'Tanh':         np.tanh(z),
    'Sign':         np.sign(z),
    'ReLU':         np.maximum(0, z)
}

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
fig.suptitle('Activation Functions', fontsize=14, fontweight='bold')

colors = ['#3498db', '#e67e22', '#2ecc71', '#9b59b6', '#e74c3c']

for ax, (name, value), color in zip(axes, activations.items(), colors):
    ax.plot(z, value, color=color, linewidth=2.5)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set_title(name, fontweight='bold', fontsize=11)
    ax.set_xlim(-4, 4)
    ax.grid(True, alpha=0.3)
    ax.set_xlabel('z')

plt.tight_layout()
plt.show()

In [ ]:
# ========================================================
# FROM-SCRATCH PERCEPTRON IMPLEMENTATION
# ========================================================
class Perceptron:
    """From-scratch implementation of the Perceptron learning algorithm described in the slides"""
    
    def __init__(self, learning_rate=0.1, max_iter=100, threshold=0.01):
        self.lr = learning_rate
        self.max_iter = max_iter
        self.threshold = threshold
        self.weights = None
        self.bias = None
        self.errors = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        # Random initial weights
        np.random.seed(42)
        self.weights = np.random.randn(n_features) * 0.01
        self.bias = 0.0
        
        for epoch in range(self.max_iter):
            total_error = 0
            for xi, yi in zip(X, y):
                y_hat = self._predict_one(xi)
                error = yi - y_hat
                # Weight update: w(k+1) = w(k) + λ(y - ŷ)x
                self.weights += self.lr * error * xi
                self.bias    += self.lr * error
                total_error  += abs(error)
            
            avg_error = total_error / n_samples
            self.errors.append(avg_error)
            
            if avg_error < self.threshold:
                print(f'  ✅ Converged! Epoch={epoch+1}, Avg Error={avg_error:.4f}')
                break
        
        return self
    
    def _predict_one(self, x):
        z = np.dot(self.weights, x) + self.bias
        return 1 if z > 0 else -1
    
    def predict(self, X):
        return np.array([self._predict_one(xi) for xi in X])


# AND gate example (linearly separable)
X_and = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_and = np.array([-1, -1, -1, 1])  # AND: True only for (1,1)

print('=== PERCEPTRON — AND GATE ===')
print('Data:', list(zip(X_and.tolist(), y_and.tolist())))

p = Perceptron(learning_rate=0.1, max_iter=50)
p.fit(X_and, y_and)

predictions = p.predict(X_and)
print(f'Predictions : {predictions}')
print(f'Ground Truth: {y_and}')
print(f'Accuracy    : {accuracy_score(y_and, predictions):.2%}')

# Learning curve
plt.figure(figsize=(8, 3))
plt.plot(p.errors, 'b-o', markersize=6)
plt.xlabel('Epoch')
plt.ylabel('Average Error')
plt.title('Perceptron Learning Curve — AND Gate')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ========================================================
# PERCEPTRON LIMITATION — XOR PROBLEM
# (Linearly Non-Separable → Fails)
# ========================================================
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([-1, 1, 1, -1])  # XOR: linearly non-separable!

print('=== PERCEPTRON LIMITATION — XOR PROBLEM ===')
p_xor = Perceptron(learning_rate=0.1, max_iter=100, threshold=0.001)
p_xor.fit(X_xor, y_xor)

preds_xor = p_xor.predict(X_xor)
print(f'Predictions : {preds_xor}')
print(f'Ground Truth: {y_xor}')
print(f'Accuracy    : {accuracy_score(y_xor, preds_xor):.2%}')
print('\n⚠️ Perceptron cannot solve XOR → Multi-layer network required!')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
problems = [('AND (Linearly Separable ✅)', X_and, y_and, p),
            ('XOR (Linearly Non-Separable ❌)', X_xor, y_xor, p_xor)]

for ax, (title, X, y, model) in zip(axes, problems):
    xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#fadbd8', '#d5f5e3']))
    ax.contour(xx, yy, Z, colors='black', linewidths=1.5)
    
    for xi, yi in zip(X, y):
        c = 'green' if yi == 1 else 'red'
        m = '^' if yi == 1 else 'o'
        ax.scatter(*xi, c=c, marker=m, s=150, edgecolors='black', zorder=5)
    
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ========================================================
# MULTI-LAYER NEURAL NETWORK — sklearn MLP
# ========================================================
print('=== MULTI-LAYER NEURAL NETWORK (MLP) — XOR Solution ===')

mlp_xor = MLPClassifier(
    hidden_layer_sizes=(4,),   # 1 hidden layer, 4 neurons
    activation='relu',
    max_iter=1000,
    random_state=42
)
mlp_xor.fit(X_xor, y_xor)
print(f'XOR Accuracy: {accuracy_score(y_xor, mlp_xor.predict(X_xor)):.2%}')
print('✅ Multi-layer network can solve the XOR problem!')

# Real classification problem
print('\n=== MLP — IRIS DATASET ===')
iris_full = load_iris()
X_full, y_full = iris_full.data, iris_full.target
X_tr, X_te, y_tr, y_te = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

architectures = [
    (50,),
    (100, 50),
    (100, 50, 25)
]

print("{:<25} {:<22} {}".format("Architecture", "Train Accuracy", "Test Accuracy"))
print('-' * 60)
for arch in architectures:
    mlp = MLPClassifier(hidden_layer_sizes=arch, max_iter=500, random_state=42)
    mlp.fit(X_tr_s, y_tr)
    train_acc = accuracy_score(y_tr, mlp.predict(X_tr_s))
    test_acc  = accuracy_score(y_te, mlp.predict(X_te_s))
    print(f"{str(arch):<25} {train_acc:.2%}               {test_acc:.2%}")

In [ ]:
# ========================================================
# DEEP NEURAL NETWORKS — VANISHING GRADIENT VISUALIZATION
# ========================================================
z_vals = np.linspace(-6, 6, 300)
sigmoid = 1 / (1 + np.exp(-z_vals))
sigmoid_grad = sigmoid * (1 - sigmoid)  # σ'(z) = σ(z)(1-σ(z))
relu = np.maximum(0, z_vals)
relu_grad = (z_vals > 0).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Vanishing Gradient Problem: Sigmoid vs ReLU', fontsize=13, fontweight='bold')

# Sigmoid
ax1 = axes[0]
ax1.plot(z_vals, sigmoid, 'b-', label='σ(z)', linewidth=2)
ax1.plot(z_vals, sigmoid_grad, 'r--', label="σ'(z) — Gradient", linewidth=2)
ax1.axhline(0.25, color='gray', linestyle=':', alpha=0.7, label='Max Gradient = 0.25')
ax1.fill_between(z_vals, sigmoid_grad, alpha=0.2, color='red')
ax1.set_title('Sigmoid: Gradient Vanishes ⚠️', fontweight='bold')
ax1.set_xlabel('z'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax1.annotate('Saturation region\n(gradient ≈ 0)', xy=(4, 0.02), fontsize=9,
             color='red', ha='center')

# ReLU
ax2 = axes[1]
ax2.plot(z_vals, relu, 'b-', label='ReLU(z)', linewidth=2)
ax2.plot(z_vals, relu_grad, 'r--', label="ReLU'(z) — Gradient", linewidth=2)
ax2.set_title('ReLU: Gradient Preserved ✅', fontweight='bold')
ax2.set_xlabel('z'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('💡 Sigmoid: Max gradient is 0.25 — vanishes multiplicatively in deep networks')
print('💡 ReLU: Gradient = 1 in positive region — gradient flow is preserved')

---
# SECTION 4: Support Vector Machines (SVM)

## 📖 Theoretical Summary

### Hyperplane
$$w^T x + b = 0$$

### Margin Maximization (Primal Problem)
$$\\min_{w,b} \\frac{\\|w\\|^2}{2} \\quad \\text{s.t.} \\quad y_i(w^T x_i + b) \\geq 1$$

### Dual Problem
$$\\max_{\\lambda} \\sum_{i=1}^n \\lambda_i - \\frac{1}{2}\\sum_{i,j} \\lambda_i \\lambda_j y_i y_j x_i^T x_j$$

### Kernel Functions
- **Polynomial:** $K(u,v) = (u^T v + 1)^p$
- **RBF:** $K(u,v) = e^{-\\|u-v\\|^2 / (2\\sigma^2)}$
- **Sigmoid:** $K(u,v) = \\tanh(ku^Tv - \\delta)$

In [ ]:
# ========================================================
# LINEAR SVM SLIDE EXAMPLE — VERIFICATION WITH SCIPY
# ========================================================
# 8 examples from the slide
X_svm = np.array([
    [0.3858, 0.4687],
    [0.4871, 0.6110],
    [0.9218, 0.4103],
    [0.7382, 0.8936],
    [0.1763, 0.0579],
    [0.4057, 0.3529],
    [0.9355, 0.8132],
    [0.2146, 0.0099]
])
y_svm = np.array([1, -1, -1, -1, 1, 1, -1, 1])

# Solve with sklearn SVM
svm_lin = SVC(kernel='linear', C=1e6)  # Large C → hard margin
svm_lin.fit(X_svm, y_svm)

w = svm_lin.coef_[0]
b = svm_lin.intercept_[0]

print('=== LINEAR SVM RESULTS ===')
print(f'Weight vector  w = [{w[0]:.4f}, {w[1]:.4f}]')
print(f'Bias term      b = {b:.4f}')
print(f'Decision boundary: {w[0]:.2f}·x₁ + {w[1]:.2f}·x₂ + {b:.2f} = 0')
print(f'\nSupport Vectors:')
for sv in svm_lin.support_vectors_:
    print(f'  {sv}')
print(f'\nSlide reference: -6.64·x₁ - 9.32·x₂ + 7.93 = 0')
print('✅ Results match!')

In [ ]:
# ========================================================
# LINEAR SVM VISUALIZATION — MARGIN AND SUPPORT VECTORS
# ========================================================
fig, ax = plt.subplots(figsize=(8, 6))

# Decision boundary and margins
x1_range = np.linspace(-0.1, 1.1, 200)

def decision_boundary(x1, w, b, offset=0):
    return (-w[0]*x1 - b + offset) / w[1]

ax.plot(x1_range, decision_boundary(x1_range, w, b),    'k-',  linewidth=2.5, label='Decision Boundary')
ax.plot(x1_range, decision_boundary(x1_range, w, b,  1), 'b--', linewidth=1.5, label='Margin (+1)')
ax.plot(x1_range, decision_boundary(x1_range, w, b, -1), 'r--', linewidth=1.5, label='Margin (-1)')

# Margin region
y_up   = decision_boundary(x1_range, w, b,  1)
y_down = decision_boundary(x1_range, w, b, -1)
ax.fill_between(x1_range, y_down, y_up, alpha=0.1, color='yellow', label='Margin Region')

# Data points
for xi, yi in zip(X_svm, y_svm):
    c = '#2ecc71' if yi == 1 else '#e74c3c'
    m = 's' if yi == 1 else 'o'
    ax.scatter(*xi, c=c, marker=m, s=100, edgecolors='black', zorder=5)

# Mark support vectors
for sv in svm_lin.support_vectors_:
    circle = plt.Circle(sv, 0.035, color='blue', fill=False, linewidth=2.5, zorder=6)
    ax.add_patch(circle)

ax.set_xlim(-0.1, 1.2); ax.set_ylim(-0.1, 1.1)
ax.set_xlabel('x₁', fontsize=12); ax.set_ylabel('x₂', fontsize=12)
ax.set_title('Linear SVM: Maximum Margin Hyperplane', fontsize=13, fontweight='bold')

green_patch = mpatches.Patch(color='#2ecc71', label='y = +1')
red_patch   = mpatches.Patch(color='#e74c3c', label='y = -1')
ax.legend(handles=[green_patch, red_patch,
          plt.Line2D([],[], color='k', label='Decision Boundary'),
          plt.Line2D([],[], color='b', linestyle='--', label='Margin ±1')],
          loc='upper left')

ax.text(0.5, 0.02, f'{w[0]:.2f}·x₁ + {w[1]:.2f}·x₂ + {b:.2f} = 0',
        fontsize=10, ha='center', transform=ax.transAxes,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.tight_layout()
plt.show()
print('🔵 Circled points = Support Vectors')

In [ ]:
# ========================================================
# NON-LINEAR SVM — KERNEL FUNCTIONS
# ========================================================
# As shown in the slide: circles in center, squares outside
np.random.seed(42)
X_circles, y_circles = make_circles(n_samples=100, noise=0.1, factor=0.4, random_state=42)
y_circles_svm = np.where(y_circles == 0, -1, 1)

kernels = {
    'Linear':      SVC(kernel='linear'),
    'Poly (p=3)':  SVC(kernel='poly', degree=3),
    'RBF (Radial)':SVC(kernel='rbf', gamma=1.0),
    'Sigmoid':     SVC(kernel='sigmoid')
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('SVM Kernel Comparison — Circle Dataset', fontsize=13, fontweight='bold')

for ax, (name, model) in zip(axes, kernels.items()):
    model.fit(X_circles, y_circles_svm)
    acc = accuracy_score(y_circles_svm, model.predict(X_circles))
    
    xx, yy = np.meshgrid(np.linspace(-1.5, 1.5, 200), np.linspace(-1.5, 1.5, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#fadbd8', '#d5f5e3']))
    ax.contour(xx, yy, Z, colors='black', linewidths=0.8)
    
    for xi, yi in zip(X_circles, y_circles_svm):
        c = '#27ae60' if yi == 1 else '#c0392b'
        ax.scatter(*xi, c=c, s=30, edgecolors='none', alpha=0.8)
    
    ax.set_title(f'{name}\nAccuracy: {acc:.2%}', fontsize=10, fontweight='bold')
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)

plt.tight_layout()
plt.show()

---
# SECTION 5: Comprehensive Comparison
## Evaluation of All Algorithms on the Same Dataset

In [ ]:
# ========================================================
# COMPARISON OF ALL ALGORITHMS
# ========================================================
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X_c, y_c = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X_c, y_c, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

models = {
    'KNN (k=5)':           KNeighborsClassifier(n_neighbors=5),
    'KNN (k=11)':          KNeighborsClassifier(n_neighbors=11),
    'Naive Bayes':         GaussianNB(),
    'MLP (1 hidden layer)':MLPClassifier(hidden_layer_sizes=(100,),    max_iter=500, random_state=42),
    'MLP (2 hidden layers)':MLPClassifier(hidden_layer_sizes=(100,50), max_iter=500, random_state=42),
    'SVM (Linear)':        SVC(kernel='linear'),
    'SVM (RBF)':           SVC(kernel='rbf')
}

results = []
print("{:<28} {:<20} {}".format("Model", "Train Accuracy", "Test Accuracy"))
print('=' * 60)

for name, model in models.items():
    model.fit(X_tr_s, y_tr)
    train_acc = accuracy_score(y_tr, model.predict(X_tr_s))
    test_acc  = accuracy_score(y_te, model.predict(X_te_s))
    results.append({'Model': name, 'Train': train_acc, 'Test': test_acc})
    print(f"{name:<28} {train_acc:.4f}              {test_acc:.4f}")

# Visual comparison
df_results = pd.DataFrame(results)

x = np.arange(len(df_results))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 5))
bars1 = ax.bar(x - width/2, df_results['Train'], width, label='Train', color='#3498db', alpha=0.85)
bars2 = ax.bar(x + width/2, df_results['Test'],  width, label='Test',  color='#e74c3c', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(df_results['Model'], rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('All Classifiers Comparison (Breast Cancer Dataset)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0.85, 1.02)
ax.grid(True, axis='y', alpha=0.4)

for bar in bars2:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ========================================================
# DECISION BOUNDARY COMPARISON (2D)
# ========================================================
X_2d, y_2d = make_moons(n_samples=200, noise=0.25, random_state=42)
X_2d_tr, X_2d_te, y_2d_tr, y_2d_te = train_test_split(X_2d, y_2d, test_size=0.2, random_state=42)

models_2d = {
    'KNN (k=3)':   KNeighborsClassifier(n_neighbors=3),
    'Naive Bayes': GaussianNB(),
    'MLP':         MLPClassifier(hidden_layer_sizes=(50,25), max_iter=500, random_state=42),
    'SVM (RBF)':   SVC(kernel='rbf', probability=True)
}

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
fig.suptitle('Decision Boundary Comparison — Half-Moon Dataset', fontsize=13, fontweight='bold')

xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2.5, 200))

for ax, (name, model) in zip(axes, models_2d.items()):
    model.fit(X_2d_tr, y_2d_tr)
    test_acc = accuracy_score(y_2d_te, model.predict(X_2d_te))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.35, cmap=ListedColormap(['#aed6f1', '#a9dfbf']))
    ax.contour(xx, yy, Z, colors='black', linewidths=1)
    
    for xi, yi in zip(X_2d, y_2d):
        c = '#2980b9' if yi == 0 else '#27ae60'
        ax.scatter(*xi, c=c, s=25, alpha=0.7)
    
    ax.set_title(f'{name}\nTest Accuracy: {test_acc:.2%}', fontsize=10, fontweight='bold')
    ax.set_xlim(-2, 3); ax.set_ylim(-1.5, 2.5)

plt.tight_layout()
plt.show()

---
# 📝 SUMMARY TABLE

| Algorithm | Type | Advantage | Disadvantage | Best Use Case |
|-----------|------|-----------|--------------|---------------|
| **KNN** | Lazy learner | Simple, interpretable | High memory/time | Small-medium data |
| **Naive Bayes** | Probabilistic | Fast, handles missing data | Independence assumption | Text classification |
| **ANN/MLP** | Neural network | Non-linear boundaries | Overfitting risk | Large, complex data |
| **SVM** | Margin-based | Good generalization, kernel trick | Slow on large data | High-dimensional data |

## 🔑 Key Concepts
- **Eager learner**: Model is built during training — Decision trees, rule-based, ANN, SVM
- **Lazy learner**: Classification is performed at prediction time — KNN
- **Overfitting**: Memorizing training data → small k (KNN), too many layers (ANN)
- **Underfitting**: Insufficient model capacity → large k (KNN), too few neurons (ANN)
- **Kernel trick**: Applying linear SVM by mapping non-linear data to a higher-dimensional space
